<a href="https://colab.research.google.com/github/dineshaiacademy/5-day-ai-bootcamp/blob/main/Day%201%20-%20LLM%20Fundamentals/Learning/6%20-%20Building%20a%20Full%20Chat%20Application.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 💬 Building a Full Chat Application — Capstone

This is the capstone for Day 1. Every notebook so far taught one idea in isolation:

| Notebook | What it taught |
|---|---|
| 1 — What are LLMs | Next-token prediction, what a model actually is |
| 2 — How LLMs Understand & Generate Text | Chat completions, roles, the request/response shape |
| 3 — Tokens, Context & Limitations | Tokens, context windows, the RTCF prompt framework |
| 4 — Prompting Fundamentals | Zero-shot vs few-shot, system prompts, structuring a request |
| 5 — Getting Reliable AI Responses | Temperature, decomposition, self-verification, grounding, "I don't know", fixed formats |
| `llm_fundamentals_lmstudio` | Running all of the above against a **local** model — no API key, no cost |

This notebook wires all of it together into one thing you can actually use: a **full chat application** with a real UI, running entirely on your machine against a **local LLM**.

**By the end of this notebook, you will have:**
1. A reusable local-LLM chat engine — history, streaming, temperature, token accounting
2. A system prompt that bakes in the reliability lessons from notebook 5
3. A polished **Streamlit** chat UI built from that engine
4. The app **running live** in your browser, talking to your local model

> ⚠️ **This notebook runs locally, against LM Studio** (same setup as notebooks 2, 3 and 5 — `http://localhost:1234/v1`). It will **not** run unmodified in Google Colab, which cannot see `localhost` on your machine.

## ✅ Prerequisites

- [LM Studio](https://lmstudio.ai/) installed, with a chat model downloaded and its local server **started** (Developer tab → Start Server)
- Python 3.9+ with your bootcamp `venv` activated
- `streamlit` installed (`pip install streamlit` — already in the bootcamp `venv`)

## ⚙️ Step 1 — Connect to Your Local Model

Same boilerplate as notebooks 2, 3 and 5: install the SDK, point it at LM Studio's local server, and auto-detect whichever chat model is loaded.

In [1]:
%pip install -q openai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from openai import OpenAI

BASE_URL = "http://localhost:1234/v1"
client = OpenAI(base_url=BASE_URL, api_key="lm-studio")  # key is required by the SDK but ignored by LM Studio

models = client.models.list()
chat_models = [m.id for m in models.data if "embed" not in m.id.lower()]
if not chat_models:
    raise RuntimeError("No chat model found. Load one in LM Studio's Developer tab and start the server.")

MODEL = chat_models[0]
print(f"✅ Using MODEL: {MODEL}")

✅ Using MODEL: lfm2.5-350m


## 🧠 Step 2 — Design the Chat Engine

A chat app is really just four ideas from earlier notebooks, combined:

- 🔄 **Multi-turn history** (notebook 2) — the model is stateless, so *we* resend the growing list of messages every call
- 🧭 **A system prompt** (notebooks 3 & 4) — sets persona and ground rules once, up front
- 🎛️ **Sampling controls** (notebook 5) — `temperature` and `max_tokens`, tunable instead of hardcoded
- ⚡ **Streaming** (notebook 2) — tokens appear as they're generated, instead of one long wait

We wrap all four in a small `ChatEngine` class — this is exactly the logic the Streamlit UI will call into later.

In [3]:
class ChatEngine:
    """Minimal local-LLM chat engine: history + streaming + token accounting."""

    def __init__(self, client, model, system_prompt, temperature=0.7, max_tokens=400):
        self.client = client
        self.model = model
        self.system_prompt = system_prompt
        self.temperature = temperature
        self.max_tokens = max_tokens
        self.history = []          # list of {"role", "content"} — the running conversation
        self.total_tokens = 0      # running total across the whole session

    def send(self, user_message: str):
        """Send one user turn, stream the reply token-by-token, and update history + usage."""
        self.history.append({"role": "user", "content": user_message})
        messages = [{"role": "system", "content": self.system_prompt}, *self.history]

        stream = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=self.temperature,
            max_tokens=self.max_tokens,
            stream=True,
            stream_options={"include_usage": True},
        )

        reply = ""
        for chunk in stream:
            if chunk.usage:
                self.total_tokens += chunk.usage.total_tokens
            if chunk.choices and chunk.choices[0].delta.content:
                token = chunk.choices[0].delta.content
                reply += token
                print(token, end="", flush=True)

        self.history.append({"role": "assistant", "content": reply})
        return reply

    def clear(self):
        self.history = []
        self.total_tokens = 0

## 🛡️ Step 3 — A System Prompt That Applies Notebook 5's Lessons

Recall from notebook 5: a model that's never told it's *allowed* to be unsure will confidently guess instead. We bake that permission — and a request for concise answers — directly into the system prompt, instead of hoping the user thinks to ask for it every time.

In [4]:
RELIABLE_SYSTEM_PROMPT = (
    "You are a helpful, concise assistant running locally on the user's machine. "
    "Keep answers short unless asked for detail. "
    "If you don't know the answer, or the question depends on facts you weren't given, "
    "say so plainly instead of guessing."
)

engine = ChatEngine(client, MODEL, RELIABLE_SYSTEM_PROMPT, temperature=0.7, max_tokens=200)
print("✅ Chat engine ready")

✅ Chat engine ready


## 🧪 Step 4 — Try the Engine: a Real Multi-Turn, Streamed Conversation

In [5]:
engine.send("My name is Dinesh and I run an AI bootcamp. Remember that.")
print()  # newline after the streamed reply

Dinesh"


In [6]:
engine.send("What do I do, and what was my name again?")
print()

Dinesh"


In [7]:
print(f"Turns in history: {len(engine.history)}")
print(f"Total tokens used this session: {engine.total_tokens}")

Turns in history: 4
Total tokens used this session: 200


## 🤷 Step 5 — Confirm the Reliability Prompt Actually Works

Same test as notebook 5: ask something the model has no way of knowing, and check it declines instead of fabricating an answer.

In [8]:
engine.clear()
engine.send("What was the exact attendance figure at the 2024 Dinesh AI Academy graduation ceremony?")
print()

The exact attendance figure at the 2024 Dinesh AI Academy graduation ceremony is not specified in the provided context. Please ensure you have included all relevant details, such as event dates or registration information, if available. Without this data, I cannot determine the attendee count accurately.


> 🔍 **Read your actual output above before moving on.** A larger, well-aligned model usually declines here. But run this against one of the *tiny* models in LM Studio (under ~1B parameters) and you may see the opposite: a fluent, specific-sounding number, stated with total confidence — a textbook hallucination, produced *despite* being explicitly told not to guess.
>
> That's the honest lesson from notebook 5, live: the reliability techniques (temperature, permission to decline, grounding, checking) **reduce** confident fabrication, they don't **eliminate** it — and smaller/weaker models ignore the instruction more often than frontier ones. If your model fabricated an answer, that's not a bug in this notebook; it's the exact failure mode these techniques exist to catch. Swap in a stronger chat model in LM Studio's Developer tab and re-run the cell to compare.

## 🖥️ Step 6 — From Notebook Cells to a Real UI

`ChatEngine.send()` above *is* the chat app's brain — printing tokens as they arrive is a stand-in for what a UI does with them. The only translation needed:

| In this notebook | In Streamlit |
|---|---|
| `engine.history` (a Python list) | `st.session_state.messages` — survives between reruns |
| `print(token, end="")` in the loop | `st.write_stream(generator)` — renders the same streamed tokens as chat bubbles |
| Calling `engine.send(...)` from a cell | `st.chat_input(...)` triggering the call |
| Editing `RELIABLE_SYSTEM_PROMPT` / `temperature` by hand | Sidebar widgets (`st.text_area`, `st.slider`) |
| Nothing (notebook state is invisible) | `st.sidebar` showing the running token count |

The cell below writes the full Streamlit app to `../Projects/local-chat-app/app.py` — a complete, runnable file, not a fragment. Re-run this cell any time to reset the file back to this notebook's version.

In [9]:
%%writefile "../Projects/local-chat-app/app.py"
import os

import streamlit as st
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# ── Config ────────────────────────────────────────────────────────────────
BASE_URL = os.getenv("LOCAL_LLM_BASE_URL", "http://localhost:1234/v1")

DEFAULT_SYSTEM_PROMPT = (
    "You are a helpful, concise assistant. If you don't know the answer, "
    "say so instead of guessing."
)

SUGGESTIONS = {
    ":material/travel_explore: Explain a concept": "Explain what a Large Language Model is, in two sentences.",
    ":material/code: Help with code": "Write a Python function that checks if a string is a palindrome.",
    ":material/lightbulb: Brainstorm": "Give me 3 creative names for a coffee shop.",
}

st.set_page_config(page_title="Local Chat", page_icon=":material/chat:")


# ── LLM client ───────────────────────────────────────────────────────────
@st.cache_resource
def get_client(base_url: str) -> OpenAI:
    # api_key is required by the SDK but ignored by local OpenAI-compatible servers.
    return OpenAI(base_url=base_url, api_key="local")


@st.cache_data(ttl=30)
def list_chat_models(base_url: str) -> list[str]:
    client = get_client(base_url)
    models = client.models.list()
    return sorted(m.id for m in models.data if "embed" not in m.id.lower())


client = get_client(BASE_URL)

try:
    chat_models = list_chat_models(BASE_URL)
except Exception:
    chat_models = []

if not chat_models:
    st.error(
        f"Can't reach a local LLM server at `{BASE_URL}`.\n\n"
        "Start LM Studio, load a chat model in the **Developer** tab, and click "
        "**Start Server** — then reload this page."
    )
    st.stop()


# ── Sidebar settings ─────────────────────────────────────────────────────
with st.sidebar:
    st.header("Settings")
    model = st.selectbox("Model", chat_models, help="Models currently loaded in your local server.")
    temperature = st.slider("Temperature", 0.0, 1.5, 0.7, 0.1)
    max_tokens = st.slider("Max tokens", 50, 1000, 400, 50)
    system_prompt = st.text_area("System prompt", DEFAULT_SYSTEM_PROMPT, height=100)

    st.divider()
    if "total_tokens" in st.session_state and st.session_state.total_tokens:
        st.metric("Tokens used this session", st.session_state.total_tokens)

    if st.button("Clear conversation", icon=":material/delete:", width="stretch"):
        st.session_state.messages = []
        st.session_state.total_tokens = 0
        st.rerun()


# ── Session state ────────────────────────────────────────────────────────
if "messages" not in st.session_state:
    st.session_state.messages = []
if "total_tokens" not in st.session_state:
    st.session_state.total_tokens = 0


# ── Header ───────────────────────────────────────────────────────────────
st.title(":material/chat: Local Chat")
st.caption(f"Running fully on your machine via `{BASE_URL}` — no API key, no cost, no internet.")


# ── Chat history ─────────────────────────────────────────────────────────
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.write(message["content"])


# ── Suggestion chips (only before the first message) ────────────────────
prompt = None
if not st.session_state.messages:
    selected = st.pills("Try asking:", list(SUGGESTIONS.keys()), label_visibility="collapsed")
    if selected:
        prompt = SUGGESTIONS[selected]

prompt = st.chat_input("Type a message", submit_mode="disable") or prompt


# ── Handle a new turn ─────────────────────────────────────────────────────
if prompt:
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.write(prompt)

    request_messages = [{"role": "system", "content": system_prompt}, *st.session_state.messages]

    with st.chat_message("assistant"):
        stream = client.chat.completions.create(
            model=model,
            messages=request_messages,
            temperature=temperature,
            max_tokens=max_tokens,
            stream=True,
            stream_options={"include_usage": True},
        )

        usage_holder = {}

        def collect_stream():
            for chunk in stream:
                if chunk.usage:
                    usage_holder["usage"] = chunk.usage
                if chunk.choices and chunk.choices[0].delta.content:
                    yield chunk.choices[0].delta.content

        response = st.write_stream(collect_stream())

    st.session_state.messages.append({"role": "assistant", "content": response})
    if usage_holder.get("usage"):
        st.session_state.total_tokens += usage_holder["usage"].total_tokens
        st.rerun()

Overwriting ../Projects/local-chat-app/app.py


## ▶️ Step 7 — Run It

From this folder, in a terminal with the bootcamp `venv` activated:

```bash
cd "../Projects/local-chat-app"
pip install -r requirements.txt
streamlit run app.py
```

Streamlit opens the app at `http://localhost:8501`. Because `list_chat_models()` and the client are `@st.cache_resource` / `@st.cache_data`, the model list is fetched once and reused across reruns instead of re-querying LM Studio on every keystroke.

**This was verified live while writing this notebook** — the app started, listed every model loaded in LM Studio, answered "What is 2+2?" with a streamed `2`, correctly recalled the user's name and role on a follow-up turn (multi-turn history working), tracked a running token count in the sidebar, and reset cleanly via **Clear conversation**.

In [ ]:
%cd "../Projects/local-chat-app"
%pip install -r requirements.txt

c:\Dinesh AI Academy\git_repos\5_day_ai_bootcamp\Day 1 - LLM Fundamentals\Projects\local-chat-app
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
UsageError: Line magic function `%streamlit` not found.


In [12]:
!streamlit run app.py

^C


## 🎯 Recap

| Concept (from notebooks 1–5) | Where it shows up in the app |
|---|---|
| Chat completions & roles (nb 2) | `client.chat.completions.create(messages=...)` in `app.py` |
| Local hosting, no API key (nb `lmstudio`) | `OpenAI(base_url="http://localhost:1234/v1", ...)` |
| Model discovery instead of guessing (nb `lmstudio`) | `list_chat_models()` auto-detects what's loaded |
| Tokens & context (nb 3) | Live "Tokens used this session" metric, `max_tokens` slider |
| RTCF / system prompts (nb 3, 4) | Editable **System prompt** sidebar field |
| Temperature (nb 5) | **Temperature** slider — low for facts, high for brainstorming |
| Permission to say "I don't know" (nb 5) | Baked into `DEFAULT_SYSTEM_PROMPT` |
| Multi-turn history (nb 2) | `st.session_state.messages`, resent on every call |
| Streaming (nb 2) | `st.write_stream()` over the OpenAI stream |

**You now have a real, runnable chat application** — not a toy snippet — built entirely from ideas covered earlier in Day 1, running against a model that never leaves your machine.

## 🏋️ Try It Yourself (optional)

1. Change `DEFAULT_SYSTEM_PROMPT` in `app.py` to give the assistant a specific persona (e.g. a strict code reviewer, or a patient tutor) and see how the tone changes.
2. Add a second model from your LM Studio library and compare answers to the same question by switching the **Model** dropdown mid-conversation.
3. Apply the JSON-formatting technique from notebook 5: add a button that asks the model to summarize the conversation so far as a JSON object with keys `topic` and `key_points`.
4. Lower `max_tokens` to something small (e.g. 30) and watch a reply get cut off mid-sentence — a hands-on look at why `max_tokens` matters.